# Import

In [ ]:
import pandas as pd

# Charger le fichier CSV avec le bon séparateur
df = pd.read_csv("../../data/dataset.csv", sep="\\|\\|\\|", engine="python")

# Ne conserver que les colonnes utiles
df = df[['modern', 'old_french']].dropna()

# Nettoyage basique (optionnel mais recommandé)
df['modern'] = df['modern'].str.strip()
df['old_french'] = df['old_french'].str.strip()

# Vérification
print(df.sample(3))

                                              modern  \
0  Salut tout le monde ! Aujourd'hui, c'était un ...   
1  Chère Sophie,  Tu ne devineras jamais ce qui m...   
2  Bonjour à tous, ici Marcel Dupré, artisan poti...   
3  Yo, c’est Lila. Alors voilà, l’autre jour, j’é...   
4  Ah, laissez-moi vous raconter une petite histo...   

                                          old_french  \
0  Saluz tout le monde ! En ce jour, fu moult bon...   
1  Chère Sophie,  Tu ne devineras ja mie ce qui m...   
2  Bien le bon jour à tous, céans Marcel Dupré, o...   
3  Salut, c'est Lila. Lors, l'autre jour, j'estoi...   
4  Ah, laissez moy vous raconter une petite histo...   

                                       modern_prompt  \
0  Écris une petite histoire enthousiaste qui par...   
1  Écris une lettre personnelle drôle qui parle d...   
2  Écris un article de blog neutre qui parle de u...   
3  Écris une anecdote émouvant qui parle de les é...   
4  Écris une anecdote drôle qui parle de le tr

In [1]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('camembert-base')

/home/utilisateur/Documents/Simplon/projet_nlp/EULA-vaaag-/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, Trainer, TrainingArguments
from torch.utils.data import Dataset
import torch

class TranslationDataset(Dataset):
    def __init__(self, modern_texts, old_french_texts, tokenizer, max_length=128):
        self.modern_texts = modern_texts
        self.old_french_texts = old_french_texts
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.modern_texts)
    
    def __getitem__(self, idx):
        modern = self.modern_texts[idx]
        old_french = self.old_french_texts[idx]
        
        # Tokenisation
        inputs = self.tokenizer(
            modern, 
            max_length=self.max_length, 
            padding='max_length', 
            truncation=True, 
            return_tensors='pt'
        )
        
        targets = self.tokenizer(
            old_french, 
            max_length=self.max_length, 
            padding='max_length', 
            truncation=True, 
            return_tensors='pt'
        )
        
        return {
            'input_ids': inputs['input_ids'].flatten(),
            'attention_mask': inputs['attention_mask'].flatten(),
            'labels': targets['input_ids'].flatten()
        }

# Modèle basé sur mT5 ou mBERT
model = AutoModelForSeq2SeqLM.from_pretrained('google/mt5-small')

ModuleNotFoundError: No module named 'torch'